Changes:
- Filtered initial SWOT data to the open ocean and excluded the Great Lakes.
    - A pixel remains valid only when ADT, ugos, vgos, and Ro are finite.
    - Masked pixels within eight grid cells (1 degree) of invalid data.
    - Masked the rectangular Great Lakes longitude-latitude box.
- For PET, use the speed contour (ring of maximum speed) instead of the effective contour.

TODOs:
- Check PET for whether eddies seem large
- Plot predicted pigments vs. actual pigments from SDP to see if there's a pattern
- Investigate why polarity doesn't seem to have a large effect on pigment concentrations
- Try sin/cos for representing day of year (season)

Plot predicted vs. actual for Tchla, HexFuco, ButFuco, Zea, and DVChla

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, cross_val_predict

repo_root = Path("/Users/jerry/school/research/eddy-tracking")
EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
gold_fp = repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet"
gold = pd.read_parquet(gold_fp)

PIGMENTS = ["Tchla", "HexFuco", "ButFuco", "Zea", "DV_chla"]
target_cols = [f"eddy_mean_{p}" for p in PIGMENTS]
predictor_cols = [
    "polarity", "season", "movement", "age_frac",
    "amplitude_cm", "radius_km", "age_days",
    "rossby_center", "rossby_abs_mean", "rossby_min", "rossby_max",
    "gs_dist_km", "center_lat", "center_lon",
]

X = pd.get_dummies(gold[predictor_cols])
y = gold[target_cols]
groups = gold["track_id"]

rf = RandomForestRegressor(random_state=2026, n_estimators=400, max_depth=8, min_samples_leaf=3, max_features=0.5)
# GroupKFold keeps every eddy (track_id) on one side of the split, since a random
# split would leak the same eddy into train and test and inflate R^2.
gkf = GroupKFold(n_splits=5)

cv_predictions = {}  # pigment -> out-of-fold predicted values, reused by the residual map below
fig, axes = plt.subplots(1, 5, figsize=(19, 4.6))
for ax, pigment, col in zip(axes, PIGMENTS, target_cols):
    actual = y[col].to_numpy()
    pred = cross_val_predict(rf, X, y[col], groups=groups, cv=gkf, n_jobs=-1)
    cv_predictions[pigment] = pred

    ax.scatter(actual, pred, s=14, alpha=0.35, color="tab:blue", linewidths=0)
    lo = min(actual.min(), pred.min())
    hi = max(actual.max(), pred.max())
    pad = 0.05 * (hi - lo)
    lo, hi = lo - pad, hi + pad
    ax.plot([lo, hi], [lo, hi], color="0.3", linewidth=1, linestyle="--", zorder=0)

    r2 = r2_score(actual, pred)
    ax.set_title(f"{pigment}\nR$^2$={r2:.2f}", fontsize=12)
    ax.set_xlabel("Actual (mg m$^{-3}$)", fontsize=10)
    ax.set_ylabel("Predicted (mg m$^{-3}$)", fontsize=10)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal", adjustable="box")
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2, linestyle="--")

fig.suptitle("Random forest: predicted vs. actual eddy pigment concentration", fontsize=13)
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.colors as colors
import numpy as np

fig, axes = plt.subplots(1, 5, figsize=(24, 5), constrained_layout=True)
for ax, pigment, col in zip(axes, PIGMENTS, target_cols):
    residual = gold[col].to_numpy() - cv_predictions[pigment]

    # Cap the color scale at the 98th percentile so the one or two most extreme
    # eddy-days (a real bloom-like outlier for Tchla/HexFuco/ButFuco) don't wash
    # out the color for every other point; those points just saturate instead.
    vmax = np.percentile(np.abs(residual), 98)
    norm = colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
    sc = ax.scatter(
        gold["center_lon"], gold["center_lat"],
        c=residual, cmap="RdBu_r", norm=norm,
        s=16, linewidths=0,
    )

    cbar = fig.colorbar(sc, ax=ax, pad=0.02, extend="both")
    cbar.set_label("Actual - predicted (mg m$^{-3}$)", fontsize=9)
    cbar.ax.tick_params(labelsize=8)
    ax.set_title(pigment, fontsize=12)
    ax.set_xlabel("Longitude", fontsize=10)
    ax.set_ylabel("Latitude", fontsize=10)
    ax.tick_params(labelsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2, linestyle="--")

fig.suptitle("Random forest residuals (actual - predicted) by eddy center location", fontsize=13)
plt.show()

Residual size vs. distance to the Gulf Stream jet and distance to coast

In [ ]:
import geopandas as gpd
from cartopy.io import shapereader
from scipy.stats import spearmanr
from shapely.geometry import box
from shapely.ops import unary_union

# Distance to the nearest coastline (Natural Earth 10m), via an azimuthal-equidistant
# projection centered on the ROI so shapely's distance() comes out in meters.
coastline_shp = shapereader.natural_earth(resolution="10m", category="physical", name="coastline")
coastline = gpd.read_file(coastline_shp)
roi_box = box(-85, 25, -50, 48)
coastline_geom = unary_union(coastline[coastline.intersects(roi_box)].geometry)
aeqd = "+proj=aeqd +lat_0=36.5 +lon_0=-68.5 +datum=WGS84 +units=m"
coastline_proj = gpd.GeoSeries([coastline_geom], crs="EPSG:4326").to_crs(aeqd).iloc[0]
points_proj = gpd.GeoSeries(gpd.points_from_xy(gold["center_lon"], gold["center_lat"]), crs="EPSG:4326").to_crs(aeqd)
gold["dist_coast_km"] = points_proj.distance(coastline_proj).to_numpy() / 1000.0
gold["dist_gs_km"] = gold["gs_dist_km"].abs()  # unsigned distance to the jet

rows = []
rel_abs_residuals = []
for pigment, col in zip(PIGMENTS, target_cols):
    residual = gold[col].to_numpy() - cv_predictions[pigment]
    rmse = np.sqrt(np.mean(residual ** 2))
    rel_abs_residuals.append(np.abs(residual) / rmse)  # so pigments of different scale are comparable

    gs_corr, gs_p = spearmanr(np.abs(residual), gold["dist_gs_km"])
    coast_corr, coast_p = spearmanr(np.abs(residual), gold["dist_coast_km"])
    rows.append({
        "pigment": pigment,
        "spearman_vs_gs_dist": gs_corr, "p_gs": gs_p,
        "spearman_vs_coast_dist": coast_corr, "p_coast": coast_p,
    })

print(pd.DataFrame(rows).set_index("pigment").round(3).to_string())

pooled_rel_residual = np.mean(rel_abs_residuals, axis=0)


def binned_trend(x, y, n_bins=8):
    bins = pd.qcut(x, n_bins, duplicates="drop")
    grouped = pd.DataFrame({"x": x, "y": y, "bin": bins}).groupby("bin", observed=True)
    return grouped["x"].mean().to_numpy(), grouped["y"].mean().to_numpy(), grouped["y"].sem().to_numpy()


fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, dist_col, label in zip(axes, ["dist_gs_km", "dist_coast_km"], ["Distance to Gulf Stream jet (km)", "Distance to coast (km)"]):
    x_mean, y_mean, y_sem = binned_trend(gold[dist_col].to_numpy(), pooled_rel_residual)
    ax.errorbar(x_mean, y_mean, yerr=y_sem, fmt="o-", color="tab:blue", capsize=3)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel("Relative |residual| (x RMSE)", fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.2, linestyle="--")
    ax.tick_params(labelsize=9)

fig.suptitle("Model error vs. distance to Gulf Stream / coast (pooled across 5 pigments)", fontsize=12)
fig.tight_layout()
plt.show()

### Visualization of Rossby Number Map on a Particular Day

In [ ]:
from pathlib import Path
import re

import matplotlib.colors as colors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr


BG_THRESHOLD_ROSSBY = 0.1


repo_root = Path("/Users/jerry/school/research/eddy-tracking")
EXPERIMENT = "gulf_stream_pigment_influencers_20241001_20251231"
gold_fp = repo_root / "data" / EXPERIMENT / "gold" / "eddy_pigment_table.parquet"
PLOT_DATE = pd.Timestamp("2025-08-16")
swot_dir = repo_root / "data" / EXPERIMENT / "bronze" / "swot_l4"
eddy_id_dir = repo_root / "data" / EXPERIMENT / "silver" / "eddy_id"

date_re = re.compile(r"\d{8}")


def swot_date(fp: Path) -> pd.Timestamp:
    match = date_re.search(fp.name)
    if match is None:
        raise ValueError(f"No YYYYMMDD date found in {fp.name}")
    return pd.Timestamp(match.group(), tz=None)


swot_files = sorted(swot_dir.glob("*.nc"))
if not swot_files:
    raise FileNotFoundError(f"No SWOT files found in {swot_dir}")

target = PLOT_DATE.normalize()
swot_fp = min(swot_files, key=lambda fp: abs((swot_date(fp) - target).days))
swot_day = swot_date(swot_fp)

with xr.open_dataset(swot_fp) as ds:
    if "time" in ds["relative_vorticity"].dims:
        ds = ds.isel(time=0)
    lon = ds["longitude"].to_numpy()
    lat = ds["latitude"].to_numpy()
    # The source variable is named relative_vorticity, but these files already
    # store normalized relative vorticity: Rossby number, Ro = zeta / f.
    rossby = ds["relative_vorticity"].to_numpy()

vmax = np.nanmax(np.abs(rossby))
vmax = max(0.1, min(float(vmax), 1.25))
rossby_norm = colors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
background_mask = np.isfinite(rossby) & (np.abs(rossby) <= BG_THRESHOLD_ROSSBY)
background_overlay = np.ma.masked_where(~background_mask, np.ones_like(rossby))
background_cmap = colors.ListedColormap(["0.55"])

# Keep the Ro field off red/blue so the red/blue eddy contours stay legible
# (orange = positive Ro, purple = negative).
rossby_cmap = "PuOr_r"

# Effective contours of eddies py-eddy-tracker (PET) detected this same day. PET
# stores contour longitudes in 0-360; shift to the SWOT grid's -180..180 to align.
eddy_layers = [
    (eddy_id_dir / "anticyclone" / f"Anticyclonic_{swot_day:%Y-%m-%d}.nc", "red", "anticyclonic"),
    (eddy_id_dir / "cyclone" / f"Cyclonic_{swot_day:%Y-%m-%d}.nc", "blue", "cyclonic"),
]
eddy_contours = []
for fp, color, label in eddy_layers:
    if not fp.exists():
        continue
    with xr.open_dataset(fp) as eds:
        clon = eds["effective_contour_longitude"].to_numpy()
        clat = eds["effective_contour_latitude"].to_numpy()
    clon = np.where(clon > 180, clon - 360, clon)
    eddy_contours.append((clon, clat, color, label))


def overlay_eddies(ax):
    for clon, clat, color, label in eddy_contours:
        for i in range(clon.shape[0]):
            ax.plot(
                clon[i],
                clat[i],
                color=color,
                linewidth=1.5,
                label=label if i == 0 else None,
            )


fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True, sharey=True)

rossby_mesh = axes[0].pcolormesh(
    lon,
    lat,
    rossby,
    cmap=rossby_cmap,
    norm=rossby_norm,
    shading="auto",
)
rossby_cbar = fig.colorbar(rossby_mesh, ax=axes[0], pad=0.02)
rossby_cbar.set_label("Ro")
axes[0].set_title(f"Rossby number ({swot_day:%Y-%m-%d})")
axes[0].set_ylabel("Latitude")
axes[0].grid(alpha=0.25, linestyle="--")

background_base = axes[1].pcolormesh(
    lon,
    lat,
    rossby,
    cmap=rossby_cmap,
    norm=rossby_norm,
    shading="auto",
)
axes[1].pcolormesh(
    lon,
    lat,
    background_overlay,
    cmap=background_cmap,
    shading="auto",
)
background_cbar = fig.colorbar(background_base, ax=axes[1], pad=0.02)
background_cbar.set_label("Ro")
axes[1].set_title(f"Background regions (|Ro| <= {BG_THRESHOLD_ROSSBY})")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].grid(alpha=0.25, linestyle="--")

overlay_eddies(axes[0])
overlay_eddies(axes[1])
axes[0].legend(loc="upper right", fontsize=9)

fig.tight_layout()
plt.show()


### Random Forest Regression

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 3 possible targets: eddy_mean_{pigment}, eddy_mean_{pigment} / bg_mean_{pigment}, and log(eddy_mean_{pigment} / bg_mean_{pigment})
# mean_pigment, mean_pigment_ratio, log_mean_pigment_ratio
# bg_mean_{pigment} is 1 value per day, which is avg of that pigment in all of the pixels w/ |Ro| <= 0.1 on that day

models = {} # {target_type: {pigment: model}}
importances = {} # {target_type: {pigment: predictor_importances}}
gold = pd.read_parquet(gold_fp)

# Make the encoding contract explicit instead of relying on pd.get_dummies dtype inference.
binary_predictor_cols = ["polarity"] # already encoded as 0/1
categorical_predictor_cols = ["season", "movement"] # expanded into one-hot columns
numeric_predictor_cols = ["age_frac"] # passed through unchanged
predictor_cols = binary_predictor_cols + categorical_predictor_cols + numeric_predictor_cols

# ---
bg_cols = [x for x in gold.columns if x.startswith("bg_mean_")]
bad_bg_rows = gold[gold[bg_cols].le(0).any(axis=1)]
print(f"Bad bg row ct: {len(bad_bg_rows)}")

eddy_cols = [x for x in gold.columns if x.startswith("eddy_mean_")]
bad_eddy_rows = gold[gold[eddy_cols].le(0).any(axis=1)]
print(f"Bad eddy row ct: {len(bad_eddy_rows)}")

bad_rows = gold[predictor_cols].isna().any(axis=1)
print(f"Bad predictor row ct: {bad_rows.sum()}")
# ---

# eddy_mean <= 0 makes log(eddy/bg) = -inf; drop those rows so all 3 targets train on the same rows
gold = gold[gold[eddy_cols].gt(0).all(axis=1)].reset_index(drop=True)

X = pd.get_dummies(
    gold[predictor_cols],
    columns=categorical_predictor_cols,
)

col_to_groups = {} # groupby/sum importance across same original col after one-hot encoding
for group in predictor_cols:
    for col in X.columns:
        if col.startswith(group + "_") or col == group:
            col_to_groups[col] = group
    # One-hot columns are named like season_DJF; binary/numeric columns keep their original names.

# eddy_mean_{pigment}
models["mean_pigment"] = {}
importances["mean_pigment"] = {}
target_cols = [x for x in gold.columns if x.startswith("eddy_mean_")]
y = gold[target_cols]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    train_size=0.75,
    random_state=2026
)
for col in target_cols:
    model = RandomForestRegressor(
        random_state=2026,
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        max_features=0.5
    )
    model.fit(X_train, y_train[col])

    # Update model
    models["mean_pigment"].update({col: model})

    # Update feature importance
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    importances["mean_pigment"].update({col: feature_importances.groupby(col_to_groups).sum().sort_values(ascending=False)})

# eddy_mean_{pigment} / bg_mean_{pigment}
models["mean_pigment_ratio"] = {}
importances["mean_pigment_ratio"] = {}
target_cols = [x for x in gold.columns if x.startswith("eddy_mean_")]
y = pd.DataFrame({col: gold[col] / gold[col.replace("eddy_mean_", "bg_mean_")] for col in target_cols})
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    train_size=0.75,
    random_state=2026
)
for col in target_cols:
    model = RandomForestRegressor(
        random_state=2026,
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        max_features=0.5
    )
    model.fit(X_train, y_train[col])

    # Update model
    models["mean_pigment_ratio"].update({col: model})

    # Update feature importance
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    importances["mean_pigment_ratio"].update({col: feature_importances.groupby(col_to_groups).sum().sort_values(ascending=False)})

# log(eddy_mean_{pigment} / bg_mean_{pigment})
models["log_mean_pigment_ratio"] = {}
importances["log_mean_pigment_ratio"] = {}
target_cols = [x for x in gold.columns if x.startswith("eddy_mean_")]
y = pd.DataFrame({col: np.log(gold[col] / gold[col.replace("eddy_mean_", "bg_mean_")]) for col in target_cols})
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    train_size=0.75,
    random_state=2026
)
for col in target_cols:
    model = RandomForestRegressor(
        random_state=2026,
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        max_features=0.5
    )
    model.fit(X_train, y_train[col])

    # Update model
    models["log_mean_pigment_ratio"].update({col: model})

    # Update feature importance
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    importances["log_mean_pigment_ratio"].update({col: feature_importances.groupby(col_to_groups).sum().sort_values(ascending=False)})

Season seemed especially important for DV_Chla (Prochlorococcus)

![](images/2026-06-27-20-00-51.png)

In [ ]:
from sklearn.metrics import r2_score

target_cols = [x for x in gold.columns if x.startswith("eddy_mean_")]
targets = {
    "mean_pigment": gold[target_cols],
    "mean_pigment_ratio": pd.DataFrame({col: gold[col] / gold[col.replace("eddy_mean_", "bg_mean_")] for col in target_cols}),
    "log_mean_pigment_ratio": pd.DataFrame({col: np.log(gold[col] / gold[col.replace("eddy_mean_", "bg_mean_")]) for col in target_cols}),
}
# mean/ratio did better than log

for target_type, y in targets.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        train_size=0.75,
        random_state=2026
    )
    rows = {}
    for col in target_cols:
        model = models[target_type][col]
        imp = importances[target_type][col]
        row = {"r2": r2_score(y_test[col], model.predict(X_test))}
        row.update({f"importance_{p}": imp[p] for p in predictor_cols})
        rows[col.replace("eddy_mean_", "")] = row
    print(f"=== {target_type} ===")
    print(pd.DataFrame(rows).T.round(4).to_string())

- DV_chla is strongly influenced by seasonal cycles

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Make the encoding contract explicit instead of relying on pd.get_dummies dtype inference.
binary_predictor_cols = ["polarity"] # already encoded as 0/1
categorical_predictor_cols = ["season", "movement"] # expanded into one-hot columns
numeric_predictor_cols = [
    # lifecycle
    "age_frac",
    # strength, size, rotation
    "amplitude_cm", "radius_km", "age_days",
    "rossby_center", "rossby_abs_mean", "rossby_min", "rossby_max",
    # where it sits
    "gs_dist_km", "center_lat", "center_lon",
]
predictor_cols = binary_predictor_cols + categorical_predictor_cols + numeric_predictor_cols
X = pd.get_dummies(
    gold[predictor_cols],
    columns=categorical_predictor_cols,
)

col_to_groups = {} # groupby/sum importance across same original col after one-hot encoding
for group in predictor_cols:
    for col in X.columns:
        if col.startswith(group + "_") or col == group:
            col_to_groups[col] = group
    # One-hot columns are named like season_DJF; binary/numeric columns keep their original names.

target_cols = [x for x in gold.columns if x.startswith("eddy_mean_")]
targets = {
    "mean_pigment": gold[target_cols],
    "mean_pigment_ratio": pd.DataFrame({col: gold[col] / gold[col.replace("eddy_mean_", "bg_mean_")] for col in target_cols}),
    "log_mean_pigment_ratio": pd.DataFrame({col: np.log(gold[col] / gold[col.replace("eddy_mean_", "bg_mean_")]) for col in target_cols}),
}

models = {}
importances = {}
for target_type, y in targets.items():
    models[target_type] = {}
    importances[target_type] = {}
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        train_size=0.75,
        random_state=2026
    )
    rows = {}
    for col in target_cols:
        model = RandomForestRegressor(
            random_state=2026,
            n_estimators=400,
            max_depth=8,
            min_samples_leaf=3,
            max_features=0.5
        )
        model.fit(X_train, y_train[col])
        imp = pd.Series(model.feature_importances_, index=X_train.columns).groupby(col_to_groups).sum().sort_values(ascending=False)
        models[target_type][col] = model
        importances[target_type][col] = imp
        row = {"r2": r2_score(y_test[col], model.predict(X_test))}
        row.update({f"importance_{p}": imp[p] for p in predictor_cols})
        rows[col.replace("eddy_mean_", "")] = row
    print(f"=== {target_type} ===")
    print(pd.DataFrame(rows).T.round(4).to_string())

In [ ]:
from calendar import monthrange

gold = pd.read_parquet(gold_fp)
POLARITY_NAME = {1: "cyclone", 0: "anticyclone"}
MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# half-month bins laid out over the full experiment as absolute time (not collapsed into one seasonal cycle)
START_YEAR, START_MONTH, N_MONTHS = 2024, 10, 15  # Oct 2024 .. Dec 2025
months = [divmod(START_YEAR * 12 + (START_MONTH - 1) + k, 12) for k in range(N_MONTHS)]
months = [(y, m + 1) for (y, m) in months]
bins_sequence = [(y, m, h) for (y, m) in months for h in (0, 1)]
month_tick_positions = [i * 2 + 0.5 for i in range(N_MONTHS)]
month_tick_labels = [
    MONTH_ABBR[m - 1] + (f"\n{y}" if (m == 1 or k == 0) else "")
    for k, (y, m) in enumerate(months)
]


def date_to_x(ts):
    # continuous x on the half-month axis; fractional offset within a bin tracks day-of-month
    mi = (ts.year - START_YEAR) * 12 + (ts.month - START_MONTH)
    if mi < 0 or mi >= N_MONTHS:
        return np.nan
    if ts.day <= 15:
        bin_center, bin_start, bin_end = mi * 2, 1, 15
    else:
        bin_center, bin_start, bin_end = mi * 2 + 1, 16, monthrange(ts.year, ts.month)[1]
    within = (ts.day - bin_start) / (bin_end - bin_start) if bin_end > bin_start else 0.5
    return bin_center - 0.5 + within


TCHLA_OUTLIER = 1.5  # drop the few extreme eddy-dates so the shared y-axis stays readable
eddy_obs = gold[gold["eddy_mean_Tchla"] <= TCHLA_OUTLIER].copy()
eddy_obs["month"] = eddy_obs["date"].dt.month
eddy_obs["year"] = eddy_obs["date"].dt.year
eddy_obs["half"] = (eddy_obs["date"].dt.day > 15).astype(int)
eddy_obs["x"] = eddy_obs["date"].apply(date_to_x)

# collapse repeated views of one eddy within a half-month to a single mean so dense eddies don't dominate a bin
eddy_hm = (
    eddy_obs.groupby(["track_id", "polarity", "year", "month", "half"])["eddy_mean_Tchla"]
    .mean().reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5.5), sharey=True, constrained_layout=True)
rng = np.random.default_rng(42)
for ax, (pol, color) in zip(axes, [(1, "#2166ac"), (0, "#b2182b")]):
    sub = eddy_obs[eddy_obs["polarity"] == pol]
    ax.scatter(sub["x"], sub["eddy_mean_Tchla"], s=22, color=color, alpha=0.18, linewidths=0, zorder=1)

    means = np.full(len(bins_sequence), np.nan)
    lo_all = np.full(len(bins_sequence), np.nan)
    hi_all = np.full(len(bins_sequence), np.nan)
    for i, (y, m, h) in enumerate(bins_sequence):
        vals = eddy_hm.loc[
            (eddy_hm["polarity"] == pol) & (eddy_hm["year"] == y)
            & (eddy_hm["month"] == m) & (eddy_hm["half"] == h),
            "eddy_mean_Tchla",
        ].to_numpy()
        if len(vals) == 0:
            continue
        means[i] = vals.mean()
        if len(vals) >= 3:
            boots = np.array([np.mean(rng.choice(vals, size=len(vals), replace=True)) for _ in range(2000)])
            lo_all[i], hi_all[i] = np.percentile(boots, [2.5, 97.5])
        else:
            lo_all[i], hi_all[i] = vals.min(), vals.max()

    valid = ~np.isnan(means)
    x_positions = np.arange(len(bins_sequence))
    yerr = np.vstack([means[valid] - lo_all[valid], hi_all[valid] - means[valid]])
    ax.errorbar(
        x_positions[valid],
        means[valid],
        yerr=yerr,
        fmt="o-",
        color=color,
        lw=2.4,
        ms=7,
        capsize=4,
        capthick=1.6,
        elinewidth=1.6,
        markeredgecolor="white",
        markeredgewidth=0.8,
        zorder=3,
    )

    ax.set_xticks(month_tick_positions)
    ax.set_xticklabels(month_tick_labels, fontsize=9)
    ax.set_xlabel("Month", fontsize=12, labelpad=8)
    ax.set_title(POLARITY_NAME[pol].capitalize(), fontsize=13, fontweight="bold", color=color, pad=12)
    ax.tick_params(axis="both", which="major", labelsize=10, length=4, width=1.0)
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_linewidth(1.0)
    ax.grid(axis="y", alpha=0.22, linestyle="--", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_xlim(-1, N_MONTHS * 2)

axes[0].set_ylim(0, eddy_obs["eddy_mean_Tchla"].max() * 1.06)
axes[0].set_ylabel("Mean T chl-a (mg m$^{-3}$)", fontsize=12, labelpad=10)
fig.suptitle("Mean Chl-a Over Time by Polarity", fontsize=15, fontweight="bold")
plt.show()